# Pipeline Medallion — Parquet Puro (sem Delta Lake)

Pipeline completo usando apenas **Parquet** como formato de armazenamento.

| Camada | Bucket | Formato | Descricao |
|--------|--------|---------|----------|
| **Landing** | `s3a://landing/` | JSON | Dados brutos (origem) |
| **Bronze** | `s3a://bronze/` | Parquet | Dados crus + metadados de ingestao |
| **Prata** | `s3a://prata/` | Parquet | Dados limpos, tipados e deduplicados |
| **Ouro** | `s3a://ouro/` | Parquet | Agregacoes de negocio prontas para consumo |

In [24]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, current_timestamp, lit, trim, upper, lower,
    from_unixtime, count, desc
)
from pyspark.sql.types import TimestampType

# Sem configs de Delta Lake — sessao simples
spark = SparkSession \
    .builder \
    .appName("medallion-parquet") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} conectado ao cluster")

Spark 3.5.5 conectado ao cluster


26/02/27 01:19:00 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


---
## Camada Bronze

Ingestao dos dados brutos do **landing** com adicao de metadados de controle:
- `_ingestion_timestamp`: quando o dado foi ingerido
- `_source_format`: formato de origem do arquivo

Gravacao em **Parquet** com `mode("overwrite")`.

In [25]:
# =============================================
# BRONZE: Ingestao crua + metadados
# =============================================

LANDING_PATH = "s3a://landing/*.json"
BRONZE_PATH  = "s3a://bronze/device/parquet"

df_landing = spark.read \
    .format("json") \
    .option("inferSchema", "true") \
    .json(LANDING_PATH)

df_landing.show()

print(f"Registros lidos do landing: {df_landing.count()}")

df_bronze = df_landing \
    .withColumn("_ingestion_timestamp", current_timestamp()) \
    .withColumn("_source_format", lit("json"))

df_bronze.write \
    .mode("overwrite") \
    .parquet(BRONZE_PATH)

print(f"Bronze gravado em {BRONZE_PATH}")

+------------+--------------------+----+------------+-------------------+-----------------+--------------------+--------------------+-------+-------+
|build_number|dt_current_timestamp|  id|manufacturer|              model|         platform|       serial_number|                 uid|user_id|version|
+------------+--------------------+----+------------+-------------------+-----------------+--------------------+--------------------+-------+-------+
|         100|       1654630947030|2392|        Acer|         OnePlus 6T|    Windows Phone|UVr864F8zUbyYOAUd...|f622857e-f179-4f4...|   5562|     50|
|         158|       1654630947030|6874|          HP|  Google Pixel 3 XL|Windows 10 Mobile|pEekWH7zGxVITv6NT...|9bb2c744-a317-421...|   9062|    788|
|         114|       1654630947030|9767|       Apple|          iPhone SE|        Danger OS|05skEogwZlX7j6twhhXX|abc61202-8853-48c...|    176|    398|
|         442|       1654630947030|1421|        Dell|          iPhone SE|              iOS|VMTnd2mMQ

Bronze gravado em s3a://bronze/device/parquet


In [26]:
# Validacao Bronze
df_bronze_check = spark.read.parquet(BRONZE_PATH)
print(f"Bronze — total de registros: {df_bronze_check.count()}")
df_bronze_check.printSchema()
df_bronze_check.show(5, truncate=False)

Bronze — total de registros: 200
root
 |-- build_number: long (nullable = true)
 |-- dt_current_timestamp: long (nullable = true)
 |-- id: long (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- model: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- serial_number: string (nullable = true)
 |-- uid: string (nullable = true)
 |-- user_id: long (nullable = true)
 |-- version: long (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_format: string (nullable = true)

+------------+--------------------+----+------------+----------------------+-----------+------------------------------+------------------------------------+-------+-------+--------------------------+--------------+
|build_number|dt_current_timestamp|id  |manufacturer|model                 |platform   |serial_number                 |uid                                 |user_id|version|_ingestion_timestamp      |_source_format|
+------------+-----------------

---
## Camada Prata (Silver)

Limpeza e padronizacao dos dados:
- Converter `dt_current_timestamp` (epoch ms) para tipo `timestamp`
- Remover duplicatas por `id`
- Filtrar registros com `id` ou `uid` nulos
- Padronizar `manufacturer` (UPPER) e `platform` (lower)
- Renomear colunas para nomes mais claros

In [27]:
# =============================================
# PRATA: Limpeza e padronizacao
# =============================================

PRATA_PATH = "s3a://prata/device/parquet"

df_bronze_raw = spark.read.parquet(BRONZE_PATH)
bronze_count = df_bronze_raw.count()

df_prata = df_bronze_raw \
    .filter(col("id").isNotNull() & col("uid").isNotNull()) \
    .dropDuplicates(["id"]) \
    .withColumn("event_timestamp", (col("dt_current_timestamp") / 1000).cast(TimestampType())) \
    .withColumn("manufacturer", upper(trim(col("manufacturer")))) \
    .withColumn("platform", lower(trim(col("platform")))) \
    .select(
        col("id").alias("device_id"),
        col("uid"),
        col("user_id"),
        col("manufacturer"),
        col("model"),
        col("platform"),
        col("version"),
        col("build_number"),
        col("serial_number"),
        col("event_timestamp"),
        col("_ingestion_timestamp")
    )

df_prata.write \
    .mode("overwrite") \
    .parquet(PRATA_PATH)

prata_count = df_prata.count()
print(f"Bronze: {bronze_count} registros")
print(f"Prata:  {prata_count} registros")
print(f"Removidos: {bronze_count - prata_count} (duplicatas + nulos)")

Bronze: 200 registros
Prata:  200 registros
Removidos: 0 (duplicatas + nulos)


In [28]:
# Validacao Prata
df_prata_check = spark.read.parquet(PRATA_PATH)
print(f"Prata — total de registros: {df_prata_check.count()}")
df_prata_check.printSchema()
df_prata_check.show(5, truncate=False)

Prata — total de registros: 200
root
 |-- device_id: long (nullable = true)
 |-- uid: string (nullable = true)
 |-- user_id: long (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- model: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- version: long (nullable = true)
 |-- build_number: long (nullable = true)
 |-- serial_number: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)

+---------+------------------------------------+-------+------------+-------------------+-----------+-------+------------+------------------------------+-----------------------+--------------------------+
|device_id|uid                                 |user_id|manufacturer|model              |platform   |version|build_number|serial_number                 |event_timestamp        |_ingestion_timestamp      |
+---------+------------------------------------+-------+------------+-------------------+--

---
## Camada Ouro (Gold)

Agregacoes de negocio prontas para consumo pelo Dremio / Metabase:
1. **Devices por fabricante** — ranking dos maiores fabricantes
2. **Devices por plataforma** — distribuicao por SO
3. **Resumo fabricante x plataforma** — tabela cruzada

In [29]:
# =============================================
# OURO: Agregacoes de negocio
# =============================================

OURO_FABRICANTE_PATH = "s3a://ouro/device/por_fabricante/parquet"
OURO_PLATAFORMA_PATH = "s3a://ouro/device/por_plataforma/parquet"
OURO_RESUMO_PATH     = "s3a://ouro/device/fabricante_plataforma/parquet"

df_silver = spark.read.parquet(PRATA_PATH)

# --- 1. Devices por fabricante ---
df_por_fabricante = df_silver \
    .groupBy("manufacturer") \
    .agg(count("*").alias("total_devices")) \
    .orderBy(desc("total_devices"))

df_por_fabricante.write \
    .mode("append") \
    .parquet(OURO_FABRICANTE_PATH)

print("Devices por fabricante:")
df_por_fabricante.show(truncate=False)

Devices por fabricante:
+------------+-------------+
|manufacturer|total_devices|
+------------+-------------+
|HUAWEI      |28           |
|HP          |25           |
|DELL        |25           |
|ASUS        |23           |
|ONEPLUS     |22           |
|ACER        |20           |
|XIAMOMI     |20           |
|APPLE       |19           |
|LENOVO      |18           |
+------------+-------------+



In [30]:
# --- 2. Devices por plataforma ---
df_por_plataforma = df_silver \
    .groupBy("platform") \
    .agg(count("*").alias("total_devices")) \
    .orderBy(desc("total_devices"))

df_por_plataforma.write \
    .mode("append") \
    .parquet(OURO_PLATAFORMA_PATH)

print("Devices por plataforma:")
df_por_plataforma.show(truncate=False)

Devices por plataforma:
+-----------------+-------------+
|platform         |total_devices|
+-----------------+-------------+
|android os       |20           |
|ios              |19           |
|windows phone    |17           |
|windows 10 mobile|16           |
|windows 8        |15           |
|windows 10       |15           |
|firefox os       |14           |
|windows rt       |13           |
|blackberry       |13           |
|danger os        |13           |
|android          |12           |
|ubuntu touch     |12           |
|windows 8.1      |11           |
|webos            |10           |
+-----------------+-------------+



In [31]:
# --- 3. Resumo fabricante x plataforma ---
df_resumo = df_silver \
    .groupBy("manufacturer", "platform") \
    .agg(count("*").alias("total_devices")) \
    .orderBy(desc("total_devices"))

df_resumo.write \
    .mode("append") \
    .parquet(OURO_RESUMO_PATH)

print("Resumo fabricante x plataforma (top 20):")
df_resumo.show(20, truncate=False)

Resumo fabricante x plataforma (top 20):
+------------+-----------------+-------------+
|manufacturer|platform         |total_devices|
+------------+-----------------+-------------+
|HP          |windows 8        |5            |
|ASUS        |windows phone    |4            |
|APPLE       |android os       |4            |
|DELL        |firefox os       |4            |
|XIAMOMI     |windows 10 mobile|4            |
|HP          |blackberry       |4            |
|HUAWEI      |windows rt       |4            |
|DELL        |android os       |4            |
|HUAWEI      |android os       |3            |
|ONEPLUS     |blackberry       |3            |
|ACER        |ios              |3            |
|ONEPLUS     |windows phone    |3            |
|HUAWEI      |windows phone    |3            |
|XIAMOMI     |android          |3            |
|ACER        |windows 10       |3            |
|APPLE       |firefox os       |3            |
|DELL        |danger os        |3            |
|ACER        |black

---
## Resumo do Pipeline

In [32]:
# Contagem final de cada camada
landing_count = spark.read.json("s3a://landing/*.json").count()
bronze_count  = spark.read.parquet(BRONZE_PATH).count()
prata_count   = spark.read.parquet(PRATA_PATH).count()
ouro_fab      = spark.read.parquet(OURO_FABRICANTE_PATH).count()
ouro_plat     = spark.read.parquet(OURO_PLATAFORMA_PATH).count()
ouro_resumo   = spark.read.parquet(OURO_RESUMO_PATH).count()

print("=" * 50)
print("RESUMO DO PIPELINE MEDALLION (PARQUET)")
print("=" * 50)
print(f"Landing  (JSON):           {landing_count} registros")
print(f"Bronze   (Parquet raw):    {bronze_count} registros")
print(f"Prata    (Parquet limpo):  {prata_count} registros")
print(f"Ouro     - por fabricante: {ouro_fab} linhas")
print(f"Ouro     - por plataforma: {ouro_plat} linhas")
print(f"Ouro     - resumo cruzado: {ouro_resumo} linhas")
print("=" * 50)

RESUMO DO PIPELINE MEDALLION (PARQUET)
Landing  (JSON):           200 registros
Bronze   (Parquet raw):    200 registros
Prata    (Parquet limpo):  200 registros
Ouro     - por fabricante: 18 linhas
Ouro     - por plataforma: 28 linhas
Ouro     - resumo cruzado: 208 linhas


26/02/27 01:31:02 WARN StandaloneAppClient$ClientEndpoint: Connection to spark-master:7077 failed; waiting for master to reconnect...
26/02/27 01:31:02 WARN StandaloneSchedulerBackend: Disconnected from Spark cluster! Waiting for reconnection...
26/02/27 01:31:02 WARN StandaloneAppClient$ClientEndpoint: Connection to spark-master:7077 failed; waiting for master to reconnect...


In [22]:
spark.stop()
print("SparkSession encerrada.")

SparkSession encerrada.
